# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the FAIR² Kilifi Mental Health Survey dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via the Croissant schema URL below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata.to_json()

print(f"{metadata['name']}: {metadata['description']}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We use the `recordSet` property from the metadata to list each record set and its fields. All references use `@id` fields to ensure clarity and reproducibility.

In [ ]:
# Retrieve record sets by @id
record_sets = dataset.metadata.recordSets

print('Record sets found in dataset:')
for rs in record_sets:
    print(f"  RecordSet @id: {rs['@id']}  | Name: {rs.get('name','(no name)')}  | Description: {rs.get('description','(none)')}")
    print('    Fields:')
    for fld in rs.get('fields', []):
        print(f"      Field @id: {fld['@id']}  | Name: {fld.get('name','(no name)')}  | dataType: {fld.get('dataType','(unknown)')}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Please use the record set and field `@id`s from the overview.

We dynamically extract all record sets and load them into Pandas DataFrames. All subsequent operations reference record sets, fields, and columns using their `@id`.

In [ ]:
# Extract data from each record set (@id)
# Collect all recordSet @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for RecordSet @id: {record_set_id}")
    print(f"Columns: {df.columns.tolist()}")
    print(df.head(), "\n")
# Choose the first RecordSet for further analysis
main_record_set_id = record_set_ids[0]
print(f"Main record set chosen for EDA: {main_record_set_id}")
print("Columns available:", dataframes[main_record_set_id].columns.tolist())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps: filter records by criteria, normalize numeric fields, categorize/group data. All field references use their `@id`.

We demonstrate filtering, normalizing, and grouping operations using one of the numeric fields and a group field from the dataset.

In [ ]:
# Choose a numeric field and a group field using their @id
# You should use the actual @ids from the data overview
# Example: Suppose 'cr:Score_GAD7' and 'cr:Gender' are valid @ids
# Replace these with the actual ones from the previous overview step
numeric_field_id = None
group_field_id = None

# List columns to auto-select
cols = dataframes[main_record_set_id].columns.tolist()
print('Columns in main record set:', cols)
# Try to pick GAD-7, PHQ-9, PSQ or similar from the columns
for c in cols:
    if 'GAD7' in c.upper():
        numeric_field_id = c
    elif 'PHQ9' in c.upper() and not numeric_field_id:
        numeric_field_id = c
    elif 'PSQ' in c.upper() and not numeric_field_id:
        numeric_field_id = c
    elif 'gender' in c.lower():
        group_field_id = c
    elif 'education' in c.lower() and not group_field_id:
        group_field_id = c
if numeric_field_id is None:
    numeric_field_id = cols[0]
if group_field_id is None:
    group_field_id = cols[1]

print(f"Numeric field selected for EDA: {numeric_field_id}")
print(f"Group field selected for grouping: {group_field_id}")

# Filter records where numeric_field > threshold
threshold = 10
filtered_df = dataframes[main_record_set_id][dataframes[main_record_set_id][numeric_field_id] > threshold]
print(f"Filtered records with {numeric_field_id} > {threshold}:")
print(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
 ) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by group_field (if exists)
if group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Here, we plot the distribution of the selected numeric field and compare means across groups, using `matplotlib` and `seaborn`.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of numeric_field
plt.figure(figsize=(7, 4))
sns.histplot(dataframes[main_record_set_id][numeric_field_id], bins=15, kde=True)
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Compare group means
if group_field_id in dataframes[main_record_set_id].columns:
    plt.figure(figsize=(7, 4))
    sns.barplot(
        x=group_field_id,
        y=numeric_field_id,
        data=dataframes[main_record_set_id],
        ci='sd'
    )
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.show()

## 6. Conclusion
We successfully loaded and explored the FAIR² Mental Health Survey Dataset from Kilifi County using the `mlcroissant` library.

Key takeaways:
- The dataset provides multiple record sets and rich demographic and mental health indicator fields, referenced by `@id`.
- Using `mlcroissant`, we loaded metadata and records, explored schema structure, and performed basic data filtering and normalization.
- Visualizations reveal distributions and group differences in mental health assessment scores.

Next steps could include more advanced statistical analyses, cross-field correlations, and incorporation of external health outcome data for predictive modeling.